# M1 Notebook 18 — Hypothesis Testing and Effect Size

**Notebook ID:** M1_N18  
**Status:** Runnable first edition  
**Random seed:** 42

> Hypothesis tests evaluate how compatible data are with a null model. Effect sizes quantify the magnitude of the observed difference.


## 1. Learning objectives

1. Define null and alternative hypotheses.
2. Interpret test statistics and p-values correctly.
3. Perform one-sample, independent, and paired t tests.
4. Perform a one-sample proportion z test.
5. Compute Cohen-style effect sizes.
6. Conduct a permutation test.
7. Understand Type I error, Type II error, power, and multiplicity.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.statistics import (
    bonferroni_alpha,
    cohen_d_independent,
    cohen_d_one_sample,
    independent_t_test,
    one_sample_t_test,
    paired_effect_size,
    paired_t_test,
    permutation_test_mean_difference,
    proportion_z_test,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
set_seed(42)
environment_info()


## 2. Hypotheses

A null hypothesis specifies a reference model, for example

\[
H_0:\mu=\mu_0.
\]

The alternative may be

\[
H_1:\mu\neq\mu_0,
\qquad
H_1:\mu>\mu_0,
\qquad
H_1:\mu<\mu_0.
\]


## 3. Test statistic and p-value

A test statistic measures discrepancy between the data and the null hypothesis.

The p-value is the probability, under the null model, of observing a result at least as extreme as the one obtained.

It is not the probability that the null hypothesis is true.


## 4. One-sample t test

In [ ]:
sample = np.array([5.2, 5.4, 5.1, 5.3, 5.5])

t_statistic, p_value = one_sample_t_test(
    sample,
    null_mean=5.0,
    alternative="greater",
)

effect = cohen_d_one_sample(sample, null_mean=5.0)

{
    "sample_mean": sample.mean(),
    "t_statistic": t_statistic,
    "p_value": p_value,
    "Cohen_d": effect,
}


## 5. Independent-samples t test

In [ ]:
programme = np.array([72, 75, 78, 74, 80, 77, 76], dtype=float)
comparison = np.array([68, 70, 72, 69, 73, 71, 70], dtype=float)

t_statistic, p_value = independent_t_test(
    programme,
    comparison,
    equal_variance=False,
)

effect = cohen_d_independent(programme, comparison)

{
    "mean_programme": programme.mean(),
    "mean_comparison": comparison.mean(),
    "mean_difference": programme.mean() - comparison.mean(),
    "t_statistic": t_statistic,
    "p_value": p_value,
    "Cohen_d": effect,
}


The Welch t test is used here because it does not assume equal population variances.


## 6. Paired t test

In [ ]:
before = np.array([22, 25, 27, 30, 24, 29, 31, 26], dtype=float)
after = np.array([18, 21, 23, 25, 20, 24, 26, 22], dtype=float)

t_statistic, p_value = paired_t_test(
    before,
    after,
)

effect = paired_effect_size(
    before,
    after,
)

{
    "mean_before": before.mean(),
    "mean_after": after.mean(),
    "mean_change": (after - before).mean(),
    "t_statistic": t_statistic,
    "p_value": p_value,
    "paired_effect_size": effect,
}


## 7. Proportion z test

In [ ]:
successes = 60
trials = 100
null_proportion = 0.50

z_statistic, p_value = proportion_z_test(
    successes,
    trials,
    null_proportion,
    alternative="greater",
)

{
    "sample_proportion": successes/trials,
    "z_statistic": z_statistic,
    "p_value": p_value,
}


## 8. Effect size versus statistical significance

Statistical significance depends on effect magnitude, sample size, variability, and test assumptions.

Effect size focuses on practical magnitude.


In [ ]:
rng = np.random.default_rng(42)

small_n_a = rng.normal(0.0, 1.0, 20)
small_n_b = rng.normal(0.3, 1.0, 20)

large_n_a = rng.normal(0.0, 1.0, 2000)
large_n_b = rng.normal(0.3, 1.0, 2000)

rows = []

for label, a, b in [
    ("Small samples", small_n_b, small_n_a),
    ("Large samples", large_n_b, large_n_a),
]:
    statistic, p = independent_t_test(a, b)
    d = cohen_d_independent(a, b)
    rows.append({
        "scenario": label,
        "mean_difference": a.mean() - b.mean(),
        "p_value": p,
        "Cohen_d": d,
    })

pd.DataFrame(rows)


A small effect can become statistically significant with a very large sample. Statistical significance alone does not establish practical importance.


## 9. Permutation testing

In [ ]:
observed_difference, permutation_p = permutation_test_mean_difference(
    programme,
    comparison,
    repetitions=10_000,
    seed=42,
)

{
    "observed_mean_difference": observed_difference,
    "permutation_p_value": permutation_p,
}


## 10. Permutation null distribution

In [ ]:
pooled = np.concatenate([programme, comparison])
rng = np.random.default_rng(42)
null_differences = np.empty(5000)

for i in range(5000):
    shuffled = rng.permutation(pooled)
    null_differences[i] = (
        shuffled[:programme.size].mean()
        - shuffled[programme.size:].mean()
    )

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_differences, bins=40, density=True)
ax.axvline(observed_difference, linestyle="--")
ax.axvline(-observed_difference, linestyle="--")
ax.set_xlabel("Permuted mean difference")
ax.set_ylabel("Density")
ax.set_title("Permutation Null Distribution")
plt.show()


## 11. Type I and Type II errors

- Type I error: reject a true null hypothesis.
- Type II error: fail to reject a false null hypothesis.
- Power: probability of rejecting the null when a specified alternative is true.


## 12. Simulated power curve

In [ ]:
effect_sizes = np.linspace(0.0, 1.0, 11)
sample_sizes = [20, 50, 100]
power_rows = []

rng = np.random.default_rng(123)

for n in sample_sizes:
    for effect_size in effect_sizes:
        rejections = 0
        repetitions = 1000

        for _ in range(repetitions):
            control = rng.normal(0.0, 1.0, n)
            treatment = rng.normal(effect_size, 1.0, n)
            _, p = independent_t_test(
                treatment,
                control,
                equal_variance=False,
            )
            rejections += p < 0.05

        power_rows.append({
            "sample_size": n,
            "effect_size": effect_size,
            "estimated_power": rejections/repetitions,
        })

power_frame = pd.DataFrame(power_rows)
power_frame.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for n in sample_sizes:
    subset = power_frame[power_frame["sample_size"] == n]
    ax.plot(
        subset["effect_size"],
        subset["estimated_power"],
        marker="o",
        label=f"n={n} per group",
    )
ax.axhline(0.8, linestyle="--")
ax.set_xlabel("Standardized effect size")
ax.set_ylabel("Estimated power")
ax.set_title("Power Increases with Effect Size and Sample Size")
ax.legend()
plt.show()


## 13. Multiple testing

When many hypotheses are tested, false positives accumulate.

Bonferroni control uses

\[
lpha^\star
=
rac{lpha}{m}.
\]


In [ ]:
family_alpha = 0.05
tests = 10
adjusted_alpha = bonferroni_alpha(
    family_alpha,
    tests,
)

{
    "family_alpha": family_alpha,
    "number_of_tests": tests,
    "per_test_alpha": adjusted_alpha,
}


## 14. Statistics interpretation

Hypothesis tests are tools for model criticism and controlled decision rules. They should be accompanied by estimates, intervals, effect sizes, assumptions, and study design information.


## 15. AI interpretation

Testing methods support:

- A/B experiments;
- model-performance comparison;
- fairness audits;
- drift detection;
- feature significance studies;
- benchmark evaluation.

Repeated experimentation and benchmark selection can create multiplicity and selection bias.


## 16. Decision Intelligence case — Evaluating a public programme

Suppose a pilot programme aims to reduce facility processing time. Each facility is measured before and after implementation.


In [ ]:
processing_before = np.array([
    14.2, 13.8, 15.1, 16.0, 14.7,
    15.5, 13.9, 16.2, 15.0, 14.5,
])
processing_after = np.array([
    12.8, 12.5, 13.6, 14.1, 13.2,
    13.9, 12.7, 14.5, 13.4, 13.1,
])

t_statistic, p_value = paired_t_test(
    processing_before,
    processing_after,
    alternative="greater",
)

effect = paired_effect_size(
    processing_after,
    processing_before,
)

{
    "mean_reduction": float(
        np.mean(processing_before - processing_after)
    ),
    "p_value": p_value,
    "standardized_effect": effect,
}


### Interpretation

A statistically significant reduction does not prove that the programme caused the change unless the design supports causal attribution. Operational importance also depends on costs, service quality, equity, sustainability, and external conditions.


## 17. Engineering notes

- Choose hypotheses before inspecting results.
- Report exact p-values rather than only significance labels.
- Pair effect sizes with confidence intervals where possible.
- Check independence, distributional, and design assumptions.
- Permutation schemes must preserve the null structure.
- Multiple testing and optional stopping inflate error rates.


## 18. Common errors

- Interpreting the p-value as \(P(H_0\mid data)\).
- Equating failure to reject with proof of no effect.
- Ignoring effect size.
- Testing many outcomes and reporting only significant results.
- Using independent tests for paired data.
- Claiming causality from an observational comparison.


## 19. Exercises

### Level A
Explain null hypothesis, p-value, effect size, and power.

### Level B
Derive the one-sample t statistic.

### Level C
Implement a permutation test and compare it with a t test.

### Capstone
Evaluate a programme outcome using an appropriate test, effect size, confidence interval, power analysis, and explicit discussion of causal and practical limitations.


## 20. Key insight

Hypothesis tests measure evidence against a null model. Effect sizes measure magnitude. Responsible inference requires both, together with uncertainty intervals, assumptions, design quality, and substantive judgment.
